In [ ]:
import torch
from math import sqrt


def generate_prior_boxes():
    feature_maps = [38, 19, 10, 5, 3, 1] #розміри 6 карт ознак, для яких будуємо prior boxes
    min_sizes = [30, 60, 111, 162, 213, 264] #основний розмір рамок на кожній карті(в пікселях)
    max_sizes = [60, 111, 162, 213, 264, 315] #розмір наступної карти (для додаткової рамки)
    aspect_ratios = [
        [2],  # Conv4_3: 4 priors
        [2, 3],  # Conv7: 6 priors
        [2, 3],  # Conv8_2: 6 priors
        [2, 3],  # Conv9_2: 6 priors
        [2],  # Conv10_2: 4 priors
        [2]  # Conv11_2: 4 priors
    ]

    priors = []
    for map_number, map_size in enumerate(feature_maps):
        for row in range(map_size):
            for column in range(map_size):
                #центр рамки ставиться в центр кожної клітинки карти ознак
                cx = (column + 0.5) / map_size
                cy = (row + 0.5) / map_size

                s_k = min_sizes[map_number] / 300.0
                priors.append([cx, cy, s_k, s_k]) #малий квадрат

                s_k_prime = sqrt(s_k * (max_sizes[map_number] / 300.0)) #більший квадрат (закриває проміжнок між масштабами сусідніх карт)
                priors.append([cx, cy, s_k_prime, s_k_prime])

                for ar in aspect_ratios[map_number]:
                    priors.append([cx, cy, s_k * sqrt(ar), s_k / sqrt(ar)]) #широка рамка (прямокутник - площа, як у квадрата, але інша форма)
                    priors.append([cx, cy, s_k / sqrt(ar), s_k * sqrt(ar)]) #висока рамка (прямокутник)

    priors = torch.tensor(priors, dtype=torch.float32)
    priors.clamp_(min=0.0, max=1.0)
    return priors  # [8732, 4] у форматі (cx, cy, w, h)


def point_form(boxes): #з центрового формату в кутовий
    return torch.cat([boxes[:, :2] - boxes[:, 2:] / 2,
                      boxes[:, :2] + boxes[:, 2:] / 2], 1)


def center_form(boxes): #з кутового формату в центровий
    return torch.cat([(boxes[:, 2:] + boxes[:, :2]) / 2,
                      boxes[:, 2:] - boxes[:, :2]], 1)


def jaccard_overlap(box_a, box_b): #знаходження IoU під рамками з обох списків (A i B - в кутовому форматі)
    A = box_a.size(0)
    B = box_b.size(0)

    right_bottom_point = torch.min(box_a[:, 2:].unsqueeze(1).expand(A, B, 2),
                       box_b[:, 2:].unsqueeze(0).expand(A, B, 2))
    left_up_point = torch.max(box_a[:, :2].unsqueeze(1).expand(A, B, 2),
                       box_b[:, :2].unsqueeze(0).expand(A, B, 2))

    inter = torch.clamp((right_bottom_point - left_up_point), min=0)
    inter_area = inter[:, :, 0] * inter[:, :, 1]

    area_a = ((box_a[:, 2] - box_a[:, 0]) * (box_a[:, 3] - box_a[:, 1])).unsqueeze(1).expand(A, B)
    area_b = ((box_b[:, 2] - box_b[:, 0]) * (box_b[:, 3] - box_b[:, 1])).unsqueeze(0).expand(A, B)

    union = area_a + area_b - inter_area
    return inter_area / union.clamp(min=1e-10)  # захист від ділення на 0


def match(threshold, truths, priors, variances, labels, loc_t, conf_t, idx):
    #threshold - поріг IoU для позитиву
    #truths - розмічені реальні об'єкти зображення, кутовий формат
    #priors - дефолтні бокси-якорі (завжди 8732)
    #variances - [0.1, 0.2] - для масштабування правок
    #labels - класи об'єктів - [0, 19]
    #loc_t - тензор з результатами (правильні поправки, які мережа мала передбачити, щоб перетворити рамку на розмічений об'єкт)
    #conf_t - тензор з результатами (правильний клас кожжної рамки, 0 - фон, 1...20 - класи об'єктів)
    #idx - номер зображення в батчі
    if truths.numel() == 0: #якщо нема об'єктів
        loc_t[idx] = 0.0
        conf_t[idx] = 0
        return

    overlaps = jaccard_overlap(truths, point_form(priors)) #матриця перетинів кожного об'єкта з кожною первинною рамкою (N_obj * 8732)

    best_prior_overlap, best_prior_idx = overlaps.max(1, keepdim=True) #знаходить найкращий prior_box для кожного реального об'єкта
    best_truth_overlap, best_truth_idx = overlaps.max(0, keepdim=True) #знаходить найкращий реалний об'єкт для кожного prior_box
    best_truth_idx.squeeze_(0)
    best_truth_overlap.squeeze_(0)
    best_prior_idx.squeeze_(1)
    best_prior_overlap.squeeze_(1)

    best_truth_overlap.index_fill_(0, best_prior_idx, 2.0) #найкращим рамкам для кожного об'єкта штучно ставиться значення 2.0
    for j in range(best_prior_idx.size(0)):
        best_truth_idx[best_prior_idx[j]] = j


    #best_truth_index - вектор з 8732 чисел, номери об'єктів, до яких прив'язаний відповідний prior_box
    matches = truths[best_truth_idx] #отримує для кожного prior_box його відповідний об'єкт (найкращий за IoU) - 4 координати
    conf = labels[best_truth_idx] + 1 #отримує для кожного prior_box відповідний клас
    conf[best_truth_overlap < threshold] = 0 #до цього кожен prior_box отримав клас відповідного найкращого об'єкта. Цей рядок залишає класи тільки в випадку перетину IoU >= 0.5, всі інші варіанти - фон


    #тут рахуються параметри, на скільки треба змінити рамки, щоб підігнати їх під відповідні об'єкти
    g_cxcy = (matches[:, :2] + matches[:, 2:]) / 2 - priors[:, :2]
    g_cxcy /= (variances[0] * priors[:, 2:])
    g_wh = (matches[:, 2:] - matches[:, :2]) / priors[:, 2:]
    g_wh = torch.log(torch.clamp(g_wh, min=1e-10)) / variances[1]
    loc = torch.cat([g_cxcy, g_wh], 1) #Отримання 4 значень (зміщення центру по ширині і висоті, а також зміну розмірів)
    #саме ці значення буде передбачати нейромережа

    loc_t[idx] = loc #цільові попраки для кожної рамки
    conf_t[idx] = conf #цільовий клас для кожної рамки


#Перетворює координати рамки (prior_box) в координати мережі через поправки, які спрогнозувала мережа (на інференсі)
def decode(loc, priors, variances=[0.1, 0.2]):
    boxes = torch.cat([
        priors[:, :2] + loc[:, :2] * variances[0] * priors[:, 2:],
        priors[:, 2:] * torch.exp(loc[:, 2:] * variances[1])
    ], 1) #отримання координат боксів в центровій формі
    boxes = point_form(boxes) #переведення в кутову форму
    return boxes.clamp(0.0, 1.0)


def nms(boxes, scores, overlap=0.45, top_k=200):
    #boxes - рамки одного класу
    #scores - впевненість кожної рамки
    #overlap - поріг перекриття, після якого рамки вважаються дублікатами
    #top_k - скільки найвпевненіших рамок розглядати
    keep = []
    if boxes.numel() == 0:
        return torch.tensor(keep, dtype=torch.long)

    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]
    area = (x2 - x1) * (y2 - y1)
    _, idx = scores.sort(0) #індекси рамок, відсортовані за впевненістю прогнозів
    idx = idx[-top_k:]

    while idx.numel() > 0:
        i = idx[-1].item() #беремо рамку з найбільшою впевненістю
        keep.append(i)
        if idx.size(0) == 1:
            break
        idx = idx[:-1] #прибираємо її зі списку

        xx1 = torch.clamp(x1[idx], min=x1[i])
        yy1 = torch.clamp(y1[idx], min=y1[i])
        xx2 = torch.clamp(x2[idx], max=x2[i])
        yy2 = torch.clamp(y2[idx], max=y2[i])
        #знаходимо координати вершин перетину

        w = torch.clamp(xx2 - xx1, min=0.0)
        h = torch.clamp(yy2 - yy1, min=0.0)
        inter = w * h

        union = area[idx] + area[i] - inter
        iou = inter / union.clamp(min=1e-10)  # захист від ділення на 0

        idx = idx[iou <= overlap] #видалення дублікатів
    return torch.tensor(keep, dtype=torch.long)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.hub import load_state_dict_from_url


class L2Norm(nn.Module):
    def __init__(self, n_channels=512, scale=20.0):
        super(L2Norm, self).__init__()
        self.n_channels = n_channels
        self.gamma = scale
        self.eps = 1e-10
        self.weight = nn.Parameter(torch.empty(self.n_channels))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.constant_(self.weight, self.gamma)

    def forward(self, x):
        norm = x.pow(2).sum(dim=1, keepdim=True).sqrt() + self.eps
        x = torch.div(x, norm)
        return self.weight.view(1, -1, 1, 1) * x

In [ ]:
class VGG16Backbone(nn.Module):
    def __init__(self, pretrained=True):
        super(VGG16Backbone, self).__init__()

        # Блок 1 (300x300 -> 150x150)
        self.conv1_1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv1_2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Блок 2 (150x150 -> 75x75)
        self.conv2_1 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv2_2 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Блок 3 (75x75 -> 38x38 завдяки ceil_mode=True)
        self.conv3_1 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.conv3_2 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.conv3_3 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True)

        # Блок 4 (вихід для Map 1: Conv4_3)
        self.conv4_1 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.conv4_2 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv4_3 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True)  # 38x38 -> 19x19

        # Блок 5 (19x19 -> 19x19)
        self.conv5_1 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv5_2 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv5_3 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.pool5 = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)

        # FC6 -> Conv6 (Dilated/Atrous згортка з dilation=6)
        self.conv6 = nn.Conv2d(512, 1024, kernel_size=3, padding=6, dilation=6)
        # FC7 -> Conv7 (вихід для Map 2: Conv7)
        self.conv7 = nn.Conv2d(1024, 1024, kernel_size=1)

        self._init_weights() #Ініціалізація Ксавьєра
        if pretrained:
            self._load_pretrained_vgg()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)

    def _load_pretrained_vgg(self): #завантаження ваг претренованої моделі на ImageNet
        vgg16_url = 'https://download.pytorch.org/models/vgg16-397923af.pth'
        state_dict = load_state_dict_from_url(vgg16_url, progress=True)

        # Індекси conv-шарів у torchvision vgg16.features
        vgg_layers = [
            (self.conv1_1, 0), (self.conv1_2, 2),
            (self.conv2_1, 5), (self.conv2_2, 7),
            (self.conv3_1, 10), (self.conv3_2, 12), (self.conv3_3, 14),
            (self.conv4_1, 17), (self.conv4_2, 19), (self.conv4_3, 21),
            (self.conv5_1, 24), (self.conv5_2, 26), (self.conv5_3, 28),
        ]
        for layer, k in vgg_layers:
            layer.weight.data.copy_(state_dict[f'features.{k}.weight'])
            layer.bias.data.copy_(state_dict[f'features.{k}.bias'])

        # fc6 (4096 x 25088) -> conv6 (1024 x 512 x 3 x 3) через децимацію:
        # кожен 4-й вихідний канал і кожен 3-й елемент ядра 7x7 (компенсується dilation=6)
        fc6_w = state_dict['classifier.0.weight'].view(4096, 512, 7, 7)
        fc6_b = state_dict['classifier.0.bias']
        self.conv6.weight.data.copy_(fc6_w[::4, :, ::3, ::3])
        self.conv6.bias.data.copy_(fc6_b[::4])

        # fc7 (4096 x 4096) -> conv7 (1024 x 1024 x 1 x 1)
        fc7_w = state_dict['classifier.3.weight'].view(4096, 4096, 1, 1)
        fc7_b = state_dict['classifier.3.bias']
        self.conv7.weight.data.copy_(fc7_w[::4, ::4])
        self.conv7.bias.data.copy_(fc7_b[::4])

    def forward(self, x):
        x = F.relu(self.conv1_1(x), inplace=True)
        x = F.relu(self.conv1_2(x), inplace=True)
        x = self.pool1(x)

        x = F.relu(self.conv2_1(x), inplace=True)
        x = F.relu(self.conv2_2(x), inplace=True)
        x = self.pool2(x)

        x = F.relu(self.conv3_1(x), inplace=True)
        x = F.relu(self.conv3_2(x), inplace=True)
        x = F.relu(self.conv3_3(x), inplace=True)
        x = self.pool3(x)

        x = F.relu(self.conv4_1(x), inplace=True)
        x = F.relu(self.conv4_2(x), inplace=True)
        conv4_3_out = F.relu(self.conv4_3(x), inplace=True)  # Map 1 (38x38)

        x = self.pool4(conv4_3_out)
        x = F.relu(self.conv5_1(x), inplace=True)
        x = F.relu(self.conv5_2(x), inplace=True)
        x = F.relu(self.conv5_3(x), inplace=True)
        x = self.pool5(x)

        x = F.relu(self.conv6(x), inplace=True)
        conv7_out = F.relu(self.conv7(x), inplace=True)  # Map 2 (19x19)

        return conv4_3_out, conv7_out


In [ ]:
class SSD300(nn.Module):
    def __init__(self, num_classes=21, pretrained_vgg=True):
        super(SSD300, self).__init__()
        self.num_classes = num_classes

        self.backbone = VGG16Backbone(pretrained=pretrained_vgg)
        self.l2_norm = L2Norm(512, scale=20.0)

        # Map 3: 10x10
        self.conv8_1 = nn.Conv2d(1024, 256, kernel_size=1)
        self.conv8_2 = nn.Conv2d(256, 512, kernel_size=3, stride=2, padding=1)

        # Map 4: 5x5
        self.conv9_1 = nn.Conv2d(512, 128, kernel_size=1)
        self.conv9_2 = nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1)

        # Map 5: 3x3
        self.conv10_1 = nn.Conv2d(256, 128, kernel_size=1)
        self.conv10_2 = nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=0)

        # Map 6: 1x1
        self.conv11_1 = nn.Conv2d(256, 128, kernel_size=1)
        self.conv11_2 = nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=0)

        source_channels = [512, 1024, 512, 256, 256, 256]
        num_priors = [4, 6, 6, 6, 4, 4]

        self.loc_heads = nn.ModuleList([
            nn.Conv2d(c, n * 4, kernel_size=3, padding=1)
            for c, n in zip(source_channels, num_priors)
        ])
        self.conf_heads = nn.ModuleList([
            nn.Conv2d(c, n * self.num_classes, kernel_size=3, padding=1)
            for c, n in zip(source_channels, num_priors)
        ])

        self._init_extra_weights()

    def _init_extra_weights(self):
        extra = [self.conv8_1, self.conv8_2, self.conv9_1, self.conv9_2,
                 self.conv10_1, self.conv10_2, self.conv11_1, self.conv11_2]
        for m in extra + list(self.loc_heads) + list(self.conf_heads):
            nn.init.xavier_uniform_(m.weight)
            nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        sources = []
        loc = []
        conf = []

        conv4_3, conv7 = self.backbone(x)
        sources.append(self.l2_norm(conv4_3))
        sources.append(conv7)

        x = F.relu(self.conv8_1(conv7), inplace=True)
        x = F.relu(self.conv8_2(x), inplace=True)
        sources.append(x)

        x = F.relu(self.conv9_1(x), inplace=True)
        x = F.relu(self.conv9_2(x), inplace=True)
        sources.append(x)

        x = F.relu(self.conv10_1(x), inplace=True)
        x = F.relu(self.conv10_2(x), inplace=True)
        sources.append(x)

        x = F.relu(self.conv11_1(x), inplace=True)
        x = F.relu(self.conv11_2(x), inplace=True)
        sources.append(x)

        for feat, l_head, c_head in zip(sources, self.loc_heads, self.conf_heads):
            loc.append(l_head(feat).permute(0, 2, 3, 1).contiguous().view(feat.size(0), -1, 4))
            conf.append(c_head(feat).permute(0, 2, 3, 1).contiguous().view(feat.size(0), -1, self.num_classes))

        loc = torch.cat(loc, 1)  # [B, 8732, 4]
        conf = torch.cat(conf, 1)  # [B, 8732, num_classes]

        return loc, conf

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class MultiBoxLoss(nn.Module):
    def __init__(self, num_classes=21, overlap_thresh=0.5, neg_pos_ratio=3):
        super(MultiBoxLoss, self).__init__()
        self.num_classes = num_classes
        self.threshold = overlap_thresh
        self.negpos_ratio = neg_pos_ratio
        self.variances = [0.1, 0.2]

    def forward(self, predictions, priors, targets):
        loc_data, conf_data = predictions  # [B, P, 4], [B, P, C]
        num = loc_data.size(0)
        num_priors = priors.size(0)
        device = loc_data.device

        # --- 1. Зіставлення priors з ground truth ---
        loc_t = torch.zeros(num, num_priors, 4, dtype=torch.float32, device=device)
        conf_t = torch.zeros(num, num_priors, dtype=torch.long, device=device)

        for idx in range(num):
            truths = targets[idx][:, :-1].to(device)
            labels = targets[idx][:, -1].to(device).long()
            match(self.threshold, truths, priors, self.variances, labels, loc_t, conf_t, idx)

        pos = conf_t > 0  # [B, P]
        num_pos = pos.sum(dim=1, keepdim=True)  # [B, 1]

        # --- 2. Localization loss (тільки позитиви) ---
        loss_l = F.smooth_l1_loss(loc_data[pos], loc_t[pos], reduction='sum')

        # --- 3. Hard negative mining (без градієнтів — лише для вибору індексів) ---
        with torch.no_grad():
            mining_loss = F.cross_entropy(
                conf_data.view(-1, self.num_classes), conf_t.view(-1), reduction='none'
            ).view(num, -1)
            mining_loss[pos] = 0.0  # позитиви не беруть участі у відборі

            _, loss_idx = mining_loss.sort(1, descending=True)
            _, idx_rank = loss_idx.sort(1)  # ранг кожного prior за величиною втрат

            num_neg = torch.clamp(self.negpos_ratio * num_pos, max=num_priors - 1)
            neg = idx_rank < num_neg  # [B, P]

        # --- 4. Confidence loss (позитиви + відібрані негативи) ---
        selected = pos | neg
        loss_c = F.cross_entropy(conf_data[selected], conf_t[selected], reduction='sum')

        # --- 5. Нормалізація на кількість позитивів ---
        N = num_pos.sum().clamp(min=1).float()
        return loss_l / N, loss_c / N

In [ ]:
import os
import random
import xml.etree.ElementTree as ET
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import torchvision
import torchvision.transforms.functional as FT

VOC_CLASSES = (
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
)
CLASS_TO_IDX = {cls_name: idx for idx, cls_name in enumerate(VOC_CLASSES)}
NUM_CLASSES = len(VOC_CLASSES) + 1  # +1 фон

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]


# ===================== Аугментації (змінюють і зображення, і бокси) =====================

def photometric_distort(image):
    """Випадкові зміни яскравості, контрасту, насиченості, відтінку (PIL)."""
    distortions = [
        lambda img: FT.adjust_brightness(img, random.uniform(0.875, 1.125)),
        lambda img: FT.adjust_contrast(img, random.uniform(0.5, 1.5)),
        lambda img: FT.adjust_saturation(img, random.uniform(0.5, 1.5)),
        lambda img: FT.adjust_hue(img, random.uniform(-18 / 255, 18 / 255)),
    ]
    random.shuffle(distortions)
    for d in distortions:
        if random.random() < 0.5:
            image = d(image)
    return image


def expand(image, boxes, max_scale=4.0):
    """Zoom out: кладемо зображення на більше полотно, заповнене середнім кольором.
    Робить об'єкти меншими -> модель вчиться знаходити малі об'єкти."""
    _, h, w = image.shape
    scale = random.uniform(1.0, max_scale)
    new_h, new_w = int(h * scale), int(w * scale)

    filler = torch.tensor(MEAN, dtype=image.dtype).view(3, 1, 1)
    new_image = filler.repeat(1, new_h, new_w)

    left = random.randint(0, new_w - w)
    top = random.randint(0, new_h - h)
    new_image[:, top:top + h, left:left + w] = image

    boxes = boxes + torch.tensor([left, top, left, top], dtype=boxes.dtype)
    return new_image, boxes


def random_crop(image, boxes, labels):
    """Випадковий кроп за схемою SSD: кроп повинен мати IoU >= min_iou хоча б з одним об'єктом.
    Залишаємо лише ті бокси, центр яких потрапив у кроп."""
    if boxes.numel() == 0:
        return image, boxes, labels

    _, h, w = image.shape
    while True:
        min_iou = random.choice([None, 0.1, 0.3, 0.5, 0.7, 0.9])
        if min_iou is None:  # без кропу
            return image, boxes, labels

        for _ in range(50):
            new_w = int(random.uniform(0.3, 1.0) * w)
            new_h = int(random.uniform(0.3, 1.0) * h)
            if not 0.5 <= new_h / max(new_w, 1) <= 2.0:
                continue

            left = random.randint(0, w - new_w)
            top = random.randint(0, h - new_h)
            crop = torch.tensor([left, top, left + new_w, top + new_h], dtype=torch.float32)

            overlap = jaccard_overlap(crop.unsqueeze(0), boxes).squeeze(0)
            if overlap.max().item() < min_iou:
                continue

            centers = (boxes[:, :2] + boxes[:, 2:]) / 2
            mask = ((centers[:, 0] > crop[0]) & (centers[:, 0] < crop[2]) &
                    (centers[:, 1] > crop[1]) & (centers[:, 1] < crop[3]))
            if not mask.any():
                continue

            new_boxes = boxes[mask].clone()
            new_labels = labels[mask]
            new_boxes[:, :2] = torch.max(new_boxes[:, :2], crop[:2]) - crop[:2]
            new_boxes[:, 2:] = torch.min(new_boxes[:, 2:], crop[2:]) - crop[:2]

            return image[:, top:top + new_h, left:left + new_w], new_boxes, new_labels


def hflip(image, boxes):
    """Горизонтальне віддзеркалення."""
    w = image.shape[2]
    image = torch.flip(image, dims=[2])
    boxes = boxes.clone()
    boxes[:, [0, 2]] = w - boxes[:, [2, 0]]
    return image, boxes


def resize_and_normalize(image, boxes, size=300):
    """Ресайз до 300x300, нормалізація ImageNet, бокси -> відносні координати [0, 1]."""
    _, h, w = image.shape
    image = FT.resize(image, [size, size], antialias=True)
    image = FT.normalize(image, MEAN, STD)
    boxes = boxes / torch.tensor([w, h, w, h], dtype=torch.float32)
    return image, boxes


def train_transform(image, boxes, labels):
    image = photometric_distort(image)
    image = FT.to_tensor(image)
    if random.random() < 0.5:
        image, boxes = expand(image, boxes)
    image, boxes, labels = random_crop(image, boxes, labels)
    if random.random() < 0.5:
        image, boxes = hflip(image, boxes)
    image, boxes = resize_and_normalize(image, boxes)
    return image, boxes, labels


def test_transform(image, boxes):
    image = FT.to_tensor(image)
    return resize_and_normalize(image, boxes)


In [ ]:
# ===================== Датасет =====================

class PascalVOCDataset(Dataset):
    def __init__(self, root='./data', year='2007', image_set='trainval',
                 train=True, keep_difficult=False, download=True):
        self.voc = torchvision.datasets.VOCDetection(
            root=root, year=year, image_set=image_set, download=download
        )
        self.train = train
        self.keep_difficult = keep_difficult

    def __len__(self):
        return len(self.voc)

    def get_annotation(self, idx):
        """Повертає бокси (абсолютні пікселі), мітки і прапорці difficult — без читання картинки."""
        root = ET.parse(self.voc.annotations[idx]).getroot()
        ann = self.voc.parse_voc_xml(root)['annotation']
        objects = ann.get('object', [])
        if not isinstance(objects, list):
            objects = [objects]

        boxes, labels, difficult = [], [], []
        for obj in objects:
            name = obj['name'].strip()
            if name not in CLASS_TO_IDX:
                continue
            bb = obj['bndbox']
            # VOC рахує пікселі з 1 -> переводимо в 0-based
            boxes.append([float(bb['xmin']) - 1, float(bb['ymin']) - 1,
                          float(bb['xmax']) - 1, float(bb['ymax']) - 1])
            labels.append(CLASS_TO_IDX[name])
            difficult.append(int(obj.get('difficult', 0)))

        return (torch.tensor(boxes, dtype=torch.float32).view(-1, 4),
                torch.tensor(labels, dtype=torch.long),
                torch.tensor(difficult, dtype=torch.bool))

    def __getitem__(self, idx):
        image = Image.open(self.voc.images[idx]).convert('RGB')
        boxes, labels, difficult = self.get_annotation(idx)

        if not self.keep_difficult:
            boxes, labels = boxes[~difficult], labels[~difficult]

        if self.train:
            image, boxes, labels = train_transform(image, boxes, labels)
        else:
            image, boxes = test_transform(image, boxes)

        targets = torch.cat([boxes, labels.unsqueeze(1).float()], dim=1)  # [N, 5]
        return image, targets


def detection_collate(batch):
    images = [sample[0] for sample in batch]
    targets = [sample[1] for sample in batch]
    return torch.stack(images, 0), targets


def build_train_dataset(data_root='./data', use_voc2012=True, download=True):
    sets = [PascalVOCDataset(data_root, '2007', 'trainval', train=True, download=download)]
    if use_voc2012:
        sets.append(PascalVOCDataset(data_root, '2012', 'trainval', train=True, download=download))
    return ConcatDataset(sets) if len(sets) > 1 else sets[0]


def show_augmented_samples(dataset, n=6, output_file='augmented_samples.png'):
    """Візуальна перевірка, що аугментації правильно зсувають бокси."""
    fig, axes = plt.subplots(2, (n + 1) // 2, figsize=(4 * ((n + 1) // 2), 8))
    for ax in axes.flatten():
        img, targets = dataset[random.randrange(len(dataset))]
        img = img * torch.tensor(STD).view(3, 1, 1) + torch.tensor(MEAN).view(3, 1, 1)
        ax.imshow(img.clamp(0, 1).permute(1, 2, 0).numpy())
        for x1, y1, x2, y2, lbl in targets.tolist():
            ax.add_patch(patches.Rectangle((x1 * 300, y1 * 300), (x2 - x1) * 300, (y2 - y1) * 300,
                                           linewidth=2, edgecolor='lime', facecolor='none'))
            ax.text(x1 * 300, y1 * 300 - 3, VOC_CLASSES[int(lbl)], color='black', fontsize=8,
                    bbox=dict(facecolor='lime', alpha=0.8, pad=1, edgecolor='none'))
        ax.axis('off')
    plt.suptitle('Приклади аугментованих тренувальних зображень')
    plt.tight_layout()
    plt.savefig(output_file, dpi=150, bbox_inches='tight')
    plt.show()


# ===================== Тренування =====================

def train_ssd(num_epochs=60, batch_size=32, lr=1e-3, warmup_iters=500,
              use_voc2012=True, data_root='./data', download=True,
              num_workers=4, use_amp=True,
              checkpoint_path='./weights/ssd300_last.pth', resume=False):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    use_amp = use_amp and device.type == 'cuda'
    print(f"Тренування запущено на пристрої: {device} | AMP: {use_amp}")

    train_dataset = build_train_dataset(data_root, use_voc2012, download)
    print(f"Тренувальних зображень: {len(train_dataset)}")
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        collate_fn=detection_collate, num_workers=num_workers,
        pin_memory=(device.type == 'cuda'), drop_last=True,
        persistent_workers=(num_workers > 0)
    )

    model = SSD300(num_classes=NUM_CLASSES, pretrained_vgg=True).to(device)
    priors = generate_prior_boxes().to(device)
    criterion = MultiBoxLoss(num_classes=NUM_CLASSES, overlap_thresh=0.5, neg_pos_ratio=3)
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)

    # LR по ітераціях: лінійний warmup, далі зменшення x0.1 на 2/3 і 5/6 тренування
    total_iters = num_epochs * len(train_loader)
    milestones = [int(total_iters * 2 / 3), int(total_iters * 5 / 6)]

    def lr_lambda(it):
        if it < warmup_iters:
            return (it + 1) / warmup_iters
        return 0.1 ** sum(it >= m for m in milestones)

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    start_epoch = 0
    loc_history, conf_history, total_history = [], [], []

    if resume and os.path.exists(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        scaler.load_state_dict(ckpt['scaler'])
        start_epoch = ckpt['epoch'] + 1
        loc_history, conf_history, total_history = ckpt['history']
        print(f"Продовжуємо з епохи {start_epoch + 1}")

    os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)

    for epoch in range(start_epoch, num_epochs):
        model.train()
        running_loc, running_conf, running_total, n_steps = 0.0, 0.0, 0.0, 0

        for i, (images, targets) in enumerate(train_loader):
            images = images.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda', enabled=use_amp):
                loc_preds, conf_preds = model(images)
            # loss рахуємо у float32 для стабільності
            loss_l, loss_c = criterion((loc_preds.float(), conf_preds.float()), priors, targets)
            loss = loss_l + loss_c

            if not torch.isfinite(loss):
                print(f"Пропущено крок {i + 1}: loss = {loss.item()}")
                scheduler.step()
                continue

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            running_loc += loss_l.item()
            running_conf += loss_c.item()
            running_total += loss.item()
            n_steps += 1

            if (i + 1) % 50 == 0 or (i + 1) == len(train_loader):
                print(f"Епоха [{epoch + 1}/{num_epochs}], Крок [{i + 1}/{len(train_loader)}] | "
                      f"LR: {optimizer.param_groups[0]['lr']:.2e} | "
                      f"Loc: {running_loc / n_steps:.4f} | "
                      f"Conf: {running_conf / n_steps:.4f} | "
                      f"Total: {running_total / n_steps:.4f}")

        loc_history.append(running_loc / max(n_steps, 1))
        conf_history.append(running_conf / max(n_steps, 1))
        total_history.append(running_total / max(n_steps, 1))

        # чекпоінт після кожної епохи (на випадок обриву сесії Colab/Kaggle)
        torch.save({
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'scaler': scaler.state_dict(),
            'epoch': epoch,
            'history': (loc_history, conf_history, total_history),
        }, checkpoint_path)

    weights_path = './weights/ssd300_voc.pth'
    torch.save(model.state_dict(), weights_path)
    print(f"Фінальні ваги збережено до {weights_path}")

    plt.figure(figsize=(9, 4.5))
    epochs_axis = range(1, len(total_history) + 1)
    plt.plot(epochs_axis, total_history, label='Total MultiBox Loss', color='black', linewidth=2)
    plt.plot(epochs_axis, loc_history, label='Localization Loss (Smooth L1)', color='crimson', linestyle='--')
    plt.plot(epochs_axis, conf_history, label='Confidence Loss (Cross-Entropy)', color='royalblue', linestyle='--')
    plt.xlabel('Епоха')
    plt.ylabel('Loss')
    plt.title('Динаміка функції втрат SSD300 (Pascal VOC)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig('ssd_training_loss.png', dpi=300, bbox_inches='tight')
    plt.show()

    return model, priors


In [ ]:
# ===================== Інференс =====================

@torch.no_grad()
def predict(model, priors, pil_img, conf_threshold=0.01, nms_threshold=0.45,
            top_k=200, max_per_image=200):
    """Детекція на одному PIL-зображенні.
    Повертає бокси (пікселі оригіналу), мітки класів (0..19) та впевненості."""
    model.eval()
    device = next(model.parameters()).device
    w, h = pil_img.size

    img, _ = test_transform(pil_img, torch.zeros((0, 4)))
    loc, conf = model(img.unsqueeze(0).to(device))
    boxes = decode(loc[0].float(), priors)
    scores = F.softmax(conf[0].float(), dim=-1)

    all_boxes, all_labels, all_scores = [], [], []
    for c in range(1, NUM_CLASSES):
        mask = scores[:, c] > conf_threshold
        if not mask.any():
            continue
        c_boxes, c_scores = boxes[mask], scores[mask, c]
        keep = nms(c_boxes, c_scores, overlap=nms_threshold, top_k=top_k).to(device)
        all_boxes.append(c_boxes[keep])
        all_scores.append(c_scores[keep])
        all_labels.append(torch.full((keep.numel(),), c - 1, dtype=torch.long, device=device))

    if not all_boxes:
        return torch.zeros((0, 4)), torch.zeros((0,), dtype=torch.long), torch.zeros((0,))

    boxes = torch.cat(all_boxes)
    labels = torch.cat(all_labels)
    scores = torch.cat(all_scores)

    if scores.numel() > max_per_image:
        top = scores.topk(max_per_image).indices
        boxes, labels, scores = boxes[top], labels[top], scores[top]

    boxes = boxes * torch.tensor([w, h, w, h], dtype=torch.float32, device=device)
    return boxes.cpu(), labels.cpu(), scores.cpu()


def detect_and_draw(model, priors, image_path, conf_threshold=0.4, nms_threshold=0.45,
                    output_file='detection_result.png'):
    orig_img = Image.open(image_path).convert("RGB")
    boxes, labels, scores = predict(model, priors, orig_img,
                                    conf_threshold=conf_threshold, nms_threshold=nms_threshold)

    fig, ax = plt.subplots(1, figsize=(10, 7))
    ax.imshow(orig_img)
    for (x1, y1, x2, y2), lbl, score in zip(boxes.tolist(), labels.tolist(), scores.tolist()):
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                       linewidth=2.5, edgecolor='#00FF00', facecolor='none'))
        ax.text(x1, max(y1 - 4, 10), f"{VOC_CLASSES[lbl]}: {score:.2f}",
                bbox=dict(facecolor='#00FF00', alpha=0.8, pad=2, edgecolor='none'),
                fontsize=10, color='black', weight='bold')

    ax.axis('off')
    plt.title(f"Результат детекції SSD300 (виявлено об'єктів: {len(scores)})")
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Зображення з розміткою збережено до {output_file}")

In [ ]:
# ===================== Запуск =====================

RUN_TRAINING = True

if RUN_TRAINING:
    # 1) спершу перевіряємо аугментації очима
    show_augmented_samples(build_train_dataset('./data', use_voc2012=False, download=True))

    # 2) тренування
    trained_model, priors = train_ssd(num_epochs=60, batch_size=32, lr=1e-3,
                                      use_voc2012=True, resume=True)

    sample_img = './data/VOCdevkit/VOC2007/JPEGImages/000005.jpg'
    if os.path.exists(sample_img):
        detect_and_draw(trained_model, priors, sample_img, conf_threshold=0.4)

In [ ]:
import os
import math
import random
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch

# VOC_CLASSES, NUM_CLASSES, PascalVOCDataset, predict — визначені в клітинці з датасетом/тренуванням

CLASS_COLORS = [plt.cm.tab20(i % 20) for i in range(len(VOC_CLASSES))]


def load_trained_model(weights_path='./weights/ssd300_voc.pth'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if not os.path.exists(weights_path):
        raise FileNotFoundError(f"Файл ваг {weights_path} не знайдено! Спочатку натренуйте модель.")

    model = SSD300(num_classes=NUM_CLASSES, pretrained_vgg=False).to(device)
    model.load_state_dict(torch.load(weights_path, map_location=device, weights_only=True))
    model.eval()
    priors = generate_prior_boxes().to(device)
    print(f"Модель завантажено ({device})")
    return model, priors


def run_batch_inference(model, priors, dataset, indices, conf_threshold=0.4, nms_threshold=0.45,
                        show_gt=True, cols=2, output_grid_file='batch_detection_results.png'):
    rows = math.ceil(len(indices) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 5.5 * rows), squeeze=False)
    axes = axes.flatten()

    for ax, idx in zip(axes, indices):
        img_path = dataset.voc.images[idx]
        orig_img = Image.open(img_path).convert("RGB")
        boxes, labels, scores = predict(model, priors, orig_img,
                                        conf_threshold=conf_threshold, nms_threshold=nms_threshold)
        ax.imshow(orig_img)

        # Ground truth — біла пунктирна рамка (для порівняння з передбаченнями)
        if show_gt:
            gt_boxes, _, _ = dataset.get_annotation(idx)
            for x1, y1, x2, y2 in gt_boxes.tolist():
                ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=1.5,
                                               edgecolor='white', linestyle='--', facecolor='none'))

        # Передбачення моделі
        for (x1, y1, x2, y2), lbl, score in zip(boxes.tolist(), labels.tolist(), scores.tolist()):
            color = CLASS_COLORS[lbl]
            text_color = 'black' if sum(color[:3]) > 1.8 else 'white'
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                           linewidth=2.5, edgecolor=color, facecolor='none'))
            ax.text(x1, max(y1 - 5, 12), f"{VOC_CLASSES[lbl]} {score:.2f}",
                    bbox=dict(facecolor=color, alpha=0.85, pad=2, edgecolor='none'),
                    fontsize=9, color=text_color, weight='bold')

        ax.set_title(f"{os.path.basename(img_path)} ({len(scores)} детекцій)", fontsize=11)
        ax.axis('off')

    for ax in axes[len(indices):]:
        ax.axis('off')

    title = 'Детекції SSD300 на тестових зображеннях VOC 2007'
    if show_gt:
        title += ' (пунктир — ground truth)'
    plt.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.savefig(output_grid_file, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Колаж збережено до: {output_grid_file}")


# ===================== Запуск =====================
model, priors = load_trained_model('./weights/ssd300_voc.pth')

test_dataset = PascalVOCDataset('./data', year='2007', image_set='test',
                                train=False, keep_difficult=True, download=True)

random.seed(42)
sample_indices = random.sample(range(len(test_dataset)), 6)

run_batch_inference(model, priors, test_dataset, sample_indices,
                    conf_threshold=0.4, nms_threshold=0.45, show_gt=True,
                    output_grid_file='batch_detection_results.png')

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from tqdm.auto import tqdm

# predict, PascalVOCDataset, load_trained_model, VOC_CLASSES — з попередніх клітинок


def voc_iou(box, gts):
    """IoU одного бокса з масивом GT-боксів за конвенцією VOC (+1 піксель)."""
    ixmin = np.maximum(gts[:, 0], box[0])
    iymin = np.maximum(gts[:, 1], box[1])
    ixmax = np.minimum(gts[:, 2], box[2])
    iymax = np.minimum(gts[:, 3], box[3])
    iw = np.maximum(ixmax - ixmin + 1.0, 0.0)
    ih = np.maximum(iymax - iymin + 1.0, 0.0)
    inter = iw * ih
    union = ((box[2] - box[0] + 1.0) * (box[3] - box[1] + 1.0) +
             (gts[:, 2] - gts[:, 0] + 1.0) * (gts[:, 3] - gts[:, 1] + 1.0) - inter)
    return inter / union


def voc_ap(rec, prec, use_07_metric=True):
    """Average Precision.
    use_07_metric=True  — 11-точкова інтерполяція (офіційна метрика VOC2007, як у статті SSD);
    use_07_metric=False — площа під PR-кривою (метрика VOC2010+)."""
    if use_07_metric:
        ap = 0.0
        for t in np.arange(0.0, 1.1, 0.1):
            p = np.max(prec[rec >= t]) if np.any(rec >= t) else 0.0
            ap += p / 11.0
        return ap

    mrec = np.concatenate(([0.0], rec, [1.0]))
    mpre = np.concatenate(([0.0], prec, [0.0]))
    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = max(mpre[i - 1], mpre[i])
    i = np.where(mrec[1:] != mrec[:-1])[0]
    return np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])


@torch.no_grad()
def evaluate_voc(model, priors, dataset, iou_threshold=0.5, conf_threshold=0.01,
                 nms_threshold=0.45, max_images=None, use_07_metric=True):
    """Оцінка mAP на VOC: детекції на всіх тестових зображеннях -> AP для кожного класу."""
    num_cls = len(VOC_CLASSES)
    n = len(dataset) if max_images is None else min(max_images, len(dataset))
    device = next(model.parameters()).device

    detections = [[] for _ in range(num_cls)]   # для кожного класу: (img_id, score, box)
    gt = {}                                     # (img_id, cls) -> GT-бокси, difficult, чи вже знайдено
    npos = np.zeros(num_cls, dtype=int)         # кількість не-difficult об'єктів кожного класу
    times = []

    # --- 1. Збираємо детекції та ground truth ---
    for i in tqdm(range(n), desc='Оцінка на тестовій вибірці'):
        gt_boxes, gt_labels, gt_diff = dataset.get_annotation(i)
        for c in range(num_cls):
            m = gt_labels == c
            if m.any():
                gt[(i, c)] = {
                    'boxes': gt_boxes[m].numpy(),
                    'difficult': gt_diff[m].numpy(),
                    'detected': np.zeros(int(m.sum()), dtype=bool),
                }
                npos[c] += int((~gt_diff[m]).sum())

        img = Image.open(dataset.voc.images[i]).convert('RGB')
        if device.type == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        boxes, labels, scores = predict(model, priors, img, conf_threshold=conf_threshold,
                                        nms_threshold=nms_threshold)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)

        for b, l, s in zip(boxes.numpy(), labels.numpy(), scores.numpy()):
            detections[l].append((i, float(s), b))

    # --- 2. AP для кожного класу ---
    aps, pr_curves = np.zeros(num_cls), {}
    for c in range(num_cls):
        dets = sorted(detections[c], key=lambda d: -d[1])   # за спаданням впевненості
        tp = np.zeros(len(dets))
        fp = np.zeros(len(dets))

        for d, (img_id, _, box) in enumerate(dets):
            g = gt.get((img_id, c))
            if g is None:                         # на зображенні немає об'єктів цього класу
                fp[d] = 1
                continue
            ious = voc_iou(box, g['boxes'])
            j = ious.argmax()
            if ious[j] > iou_threshold:
                if g['difficult'][j]:             # difficult не рахуються ні як TP, ні як FP
                    continue
                if not g['detected'][j]:
                    tp[d] = 1
                    g['detected'][j] = True
                else:                             # повторна детекція того самого об'єкта
                    fp[d] = 1
            else:
                fp[d] = 1

        tp, fp = np.cumsum(tp), np.cumsum(fp)
        rec = tp / max(npos[c], 1)
        prec = tp / np.maximum(tp + fp, np.finfo(np.float64).eps)
        aps[c] = voc_ap(rec, prec, use_07_metric)
        pr_curves[c] = (rec, prec)

    warm = times[10:] if len(times) > 20 else times   # перші ітерації — прогрів GPU
    ms_per_image = 1000 * float(np.mean(warm))
    return aps, pr_curves, ms_per_image, npos


def report_results(aps, pr_curves, ms_per_image, npos, use_07_metric=True,
                   csv_file='ssd_voc2007_ap.csv'):
    mAP = aps.mean()
    metric_name = 'VOC07 11-point' if use_07_metric else 'VOC area'

    # --- Таблиця AP по класах ---
    df = pd.DataFrame({'Клас': VOC_CLASSES,
                       'Кількість об\'єктів': npos,
                       'AP, %': np.round(aps * 100, 2)})
    df = df.sort_values('AP, %', ascending=False).reset_index(drop=True)
    df.to_csv(csv_file, index=False)
    print(df.to_string(index=False))
    print(f"\nmAP@0.5 ({metric_name}): {mAP * 100:.2f}%")
    print(f"Швидкість: {ms_per_image:.1f} мс/зображення (~{1000 / ms_per_image:.1f} FPS, "
          f"batch=1, з препроцесингом і NMS)")

    print("\nДовідково (VOC2007 test, тренування на VOC07+12, з оригінальних статей):")
    print("  Fast R-CNN (VGG16)   — 70.0% mAP")
    print("  Faster R-CNN (VGG16) — 73.2% mAP")
    print("  SSD300 (оригінал)    — 74.3% / 77.2% mAP (без / з розширеною аугментацією)")

    # --- Графік AP по класах ---
    order = np.argsort(aps)
    plt.figure(figsize=(9, 7))
    bars = plt.barh(np.array(VOC_CLASSES)[order], aps[order] * 100, color='steelblue')
    plt.axvline(mAP * 100, color='crimson', linestyle='--', label=f'mAP = {mAP * 100:.1f}%')
    for bar, v in zip(bars, aps[order] * 100):
        plt.text(v + 0.5, bar.get_y() + bar.get_height() / 2, f'{v:.1f}', va='center', fontsize=8)
    plt.xlabel('AP@0.5, %')
    plt.title('SSD300: Average Precision по класах (VOC 2007 test)')
    plt.xlim(0, 105)
    plt.legend(loc='lower right')
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig('ssd_ap_per_class.png', dpi=300, bbox_inches='tight')
    plt.show()

    # --- PR-криві: 3 найкращі і 3 найгірші класи ---
    selected = list(order[-3:][::-1]) + list(order[:3])
    plt.figure(figsize=(8, 6))
    for c in selected:
        rec, prec = pr_curves[c]
        style = '-' if c in order[-3:] else '--'
        plt.plot(rec, prec, style, linewidth=2, label=f'{VOC_CLASSES[c]} (AP={aps[c] * 100:.1f}%)')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall криві (суцільні — найкращі класи, пунктир — найгірші)')
    plt.xlim(0, 1)
    plt.ylim(0, 1.02)
    plt.legend(loc='lower left', fontsize=9)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('ssd_pr_curves.png', dpi=300, bbox_inches='tight')
    plt.show()

    return df, mAP


# ===================== Запуск =====================
model, priors = load_trained_model('./weights/ssd300_voc.pth')
test_dataset = PascalVOCDataset('./data', year='2007', image_set='test',
                                train=False, keep_difficult=True, download=True)

# Для швидкої перевірки можна поставити max_images=500; для звіту — None (усі 4952 зображення)
aps, pr_curves, ms_per_image, npos = evaluate_voc(model, priors, test_dataset,
                                                  max_images=None, use_07_metric=True)
results_df, mAP = report_results(aps, pr_curves, ms_per_image, npos, use_07_metric=True)